# Diabetes Hospital Readmission Prediction

XGBoost model that predicts whether a diabetic inpatient will be readmitted, using the UCI Diabetes 130-US Hospitals dataset (101,766 encounters).

The model target is binary: readmitted at any time (`<30` or `>30` days) versus not readmitted. The EDA section also looks at the `<30` day group on its own.

Place `diabetic_data.csv` in the same folder as this notebook (see `data/README.md`). This was a team project; my part was data preprocessing and EDA.

### Load the Dataset

In [ ]:
import pandas as pd

# Load the dataset and drop the 'race' column
df = pd.read_csv("diabetic_data.csv")  # Update the path if running locally

# View the shape of the dataset
print("Shape of dataset:", df.shape)

# Display data types and non-null counts
df.info()

In [ ]:
# Preview the first 5 rows
df.head()

### Handling Missing Values: missing counts

In [ ]:
# Replace '?' strings with actual missing values
df.replace('?', pd.NA, inplace=True)

# Now check for missing values
missing_summary = df.isna().sum()

# Only show columns with missing values
print("Missing values per column:\n")
print(missing_summary[missing_summary > 0])

### Handling Missing Values: percentage missing

In [ ]:
# Calculate percentage of missing values per column
missing_percent = df.isna().sum() / len(df) * 100

# Filter only columns with missing values and sort them
missing_percent_filtered = missing_percent[missing_percent > 0].sort_values(ascending=False)

# Display the result
print("Percentage of missing values per column:\n")
print(missing_percent_filtered)

### Drop weight and fill categorical missing values

In [ ]:
# Drop the 'weight' column due to excessive missing values
df = df.drop(columns=['weight'])

# Fill categorical missing values with 'Unknown'
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')
df['payer_code'] = df['payer_code'].fillna('Unknown')
df['diag_1'] = df['diag_1'].fillna('Unknown')
df['diag_2'] = df['diag_2'].fillna('Unknown')
df['diag_3'] = df['diag_3'].fillna('Unknown')

### Relabel A1Cresult and max_glu_serum

In [ ]:
# Clean any extra whitespace from strings
df['A1Cresult'] = df['A1Cresult'].astype(str).str.strip()
df['max_glu_serum'] = df['max_glu_serum'].astype(str).str.strip()

# Replace values with descriptive labels
a1c_label_mapping = {
    'None': 'No Test',
    'Norm': 'Normal',
    '>7': 'High',
    '>8': 'High'
}

glu_label_mapping = {
    'None': 'No Test',
    'Norm': 'Normal',
    '>200': 'High',
    '>300': 'Very High'
}

df['A1Cresult'] = df['A1Cresult'].replace(a1c_label_mapping)
df['max_glu_serum'] = df['max_glu_serum'].replace(glu_label_mapping)

# Confirm changes
print("A1Cresult categories:", df['A1Cresult'].unique())
print("max_glu_serum categories:", df['max_glu_serum'].unique())

# Replace any remaining NaN with 'No Test'
df['A1Cresult'] = df['A1Cresult'].fillna('No Test')
df['max_glu_serum'] = df['max_glu_serum'].fillna('No Test')

### Check remaining missing values

In [ ]:
df.isna().sum()

### Overview of Target Variable

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set up plot style
sns.set(style="whitegrid")

# Plot readmission distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='readmitted', palette='Set2')
plt.title("Distribution of Readmission Status")
plt.xlabel("Readmitted")
plt.ylabel("Count")
plt.show()

### Age Distribution

In [ ]:
# Countplot for Age
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='age', order=sorted(df['age'].unique()), palette='coolwarm')
plt.title("Age Group Distribution")
plt.xlabel("Age Group")
plt.ylabel("Number of Patients")
plt.show()

### Age groups with the highest proportion of early readmissions

In [ ]:
# Filter dataset for <30-day readmissions
readmit_under_30 = df[df['readmitted'] == '<30']

# Count total patients and <30-day readmissions per age group
age_total = df['age'].value_counts().sort_index()
age_under_30 = readmit_under_30['age'].value_counts().sort_index()

# Calculate percentage of <30-day readmissions per age group
age_under_30_percent = (age_under_30 / age_total * 100).fillna(0)

# Plot line chart
plt.figure(figsize=(10, 6))
plt.plot(age_under_30_percent.index, age_under_30_percent.values, marker='o', linestyle='-', color='crimson')
plt.title('Percentage of Patients Readmitted Within 30 Days by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Percent Readmitted (<30 Days)')
plt.grid(True)
plt.tight_layout()
plt.show()

### Average Time in Hospital by Readmission Status

In [ ]:
# Calculate average hospital stay per readmission category
time_in_hospital_avg = df.groupby('readmitted')['time_in_hospital'].mean().sort_index()

# Plot line graph
plt.figure(figsize=(8, 5))
plt.plot(time_in_hospital_avg.index, time_in_hospital_avg.values, marker='o', linestyle='-', color='navy')
plt.title('Average Time in Hospital by Readmission Status')
plt.xlabel('Readmission Status')
plt.ylabel('Average Days in Hospital')
plt.grid(True)
plt.tight_layout()
plt.show()

### A1Cresult Distribution

In [ ]:
# Create percentage table
a1c_percent = df.groupby(['A1Cresult', 'readmitted']).size().unstack().apply(lambda x: x / x.sum() * 100, axis=1)

# Plot
a1c_percent.plot(kind='bar', stacked=True, colormap='Set2', figsize=(10, 6))
plt.title('Readmission Status Distribution by A1C Result (Percentage)')
plt.ylabel('Percentage of Patients')
plt.xlabel('A1C Result')
plt.legend(title='Readmitted')
plt.tight_layout()
plt.show()

### max_glu_serum Distribution

In [ ]:
# Group and calculate percentages
glu_percent = df.groupby(['max_glu_serum', 'readmitted']).size().unstack().apply(lambda x: x / x.sum() * 100, axis=1)

# Plot stacked percentage bar chart
glu_percent.plot(kind='bar', stacked=True, colormap='Set3', figsize=(10, 6))
plt.title('Readmission Status Distribution by Max Glucose Serum (Percentage)')
plt.ylabel('Percentage of Patients')
plt.xlabel('Max Glucose Serum')
plt.legend(title='Readmitted')
plt.tight_layout()
plt.show()

### Prediction Analysis

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv("diabetic_data.csv")

# Drop identifier and irrelevant columns
df.drop(columns=['encounter_id', 'patient_nbr', 'weight'], inplace=True, errors='ignore')

# Replace ? with NaN
df.replace('?', pd.NA, inplace=True)

# Handle missing values
for col in ['payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].fillna('Unknown')

# Map test result columns
glu_mapping = {'None': 'No Test', 'Norm': 'Normal', '>200': 'High', '>300': 'Very High'}
a1c_mapping = {'None': 'No Test', 'Norm': 'Normal', '>7': 'High', '>8': 'High'}

df['max_glu_serum'] = df['max_glu_serum'].astype(str).str.strip().replace(glu_mapping).fillna('No Test')
df['A1Cresult'] = df['A1Cresult'].astype(str).str.strip().replace(a1c_mapping).fillna('No Test')

# Convert readmitted column to binary (0 = No, 1 = <30 or >30)
df['readmitted'] = df['readmitted'].replace({'>30': 1, '<30': 1, 'NO': 0})

# Encode categorical variables
label_encoders = {}
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Define features and target
X = df.drop('readmitted', axis=1)
y = df['readmitted']

# Train-test split
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(X, y, test_size=0.2, random_state=42)

### 🔧 Hyperparameter Tuning with XGBoost and GridSearchCV

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define the parameter grid
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Set up and fit GridSearchCV
grid_xgb = GridSearchCV(
    estimator=XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    param_grid=param_grid_xgb,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1,
    verbose=1
)

# Fit to training data
grid_xgb.fit(X_train_bin, y_train_bin)

# Use the best model
best_xgb = grid_xgb.best_estimator_

# Make predictions on the test set
y_pred_xgb = best_xgb.predict(X_test_bin)

# Output performance
print("Best Parameters:", grid_xgb.best_params_)
print("Accuracy:", round(accuracy_score(y_test_bin, y_pred_xgb), 4))
print("\nClassification Report:")
print(classification_report(y_test_bin, y_pred_xgb))
print("Confusion Matrix:")
print(confusion_matrix(y_test_bin, y_pred_xgb))

                          

### Feature Importance from XGBoost

In [ ]:
# Extract feature importances from best model
importances = best_xgb.feature_importances_

# Create a DataFrame for sorting and plotting
feat_imp_df = pd.DataFrame({
    'Feature': X_train_bin.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot Top 15 Features
plt.figure(figsize=(10, 6))
plt.barh(feat_imp_df['Feature'][:15][::-1], feat_imp_df['Importance'][:15][::-1])
plt.xlabel('Feature Importance Score')
plt.title('Top 15 Important Features (XGBoost - Cleaned Data)')
plt.tight_layout()
plt.show()